In [ ]:
# Install required packages inside the notebook
%pip install anomalib matplotlib numpy opencv-python torch

# Data Normalization

In [ ]:
import os
import sys

# Ensure the current directory is in the path so local modules can be imported
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from ImageProcessor.XRayImage import XRayImage

def normalize(data_folder: str, output_directory: str):

    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    if not os.path.exists(data_folder):
        print(f"Folder ze zdjęciami do przetworzenia nie istnieje: {data_folder}")
        return


    for root, dirs, files in os.walk(data_folder):
        for file in files:
            if file.find('czarno') == -1:
                continue

            # Sprawdzamy czy to obrazek
            if file.lower().endswith('.bmp'):
                full_path = os.path.join(root, file)

                img = XRayImage(src=full_path)
                img.applyFilters()
                img.generateTiles()
                img.saveTiles(output_directory)

print("Data normalization functions defined.")

In [ ]:
# Run normalization
normalize("raw_data/czyste", "datasets/train/good")
normalize("raw_data/brudne", "datasets/test/defect")
print("Normalization finished.")

# Training with Anomalib

In [ ]:
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

def train_model():
    # 1. Konfiguracja Danych
    datamodule = Folder(
        name="my_custom_dataset",
        root="datasets",
        normal_dir="train/good",   # Folder treningowy
        abnormal_dir="test/defect", # Folder z defektami do testów
        normal_test_dir="test/good", # Folder z dobrymi próbkami do testów
        train_batch_size=32,
        eval_batch_size=32,
        # WAŻNE NA WINDOWS: Czasem warto ustawić num_workers na 0, jeśli nadal będą błędy
        # num_workers=0
    )

    # setup() też warto wywołać wewnątrz main
    datamodule.setup()

    # 2. Inicjalizacja Modelu
    model = Patchcore(
        backbone="resnet18",
        pre_trained=True
    )

    # 3. Konfiguracja Silnika (Engine)
    engine = Engine(
        accelerator="auto",
        max_epochs=1,
    )

    # 4. Trening
    print("Rozpoczynam trening...")
    engine.fit(datamodule=datamodule, model=model)

    # 5. Testowanie
    print("Rozpoczynam testy...")
    # Tutaj poprawka z poprzedniej odpowiedzi (setup dla testu), aby uniknąć błędu iter()
    datamodule.setup(stage="test")
    test_results = engine.test(datamodule=datamodule, model=model)
    print(test_results)


In [ ]:
if __name__ == "__main__":
    # Ten blok jest KLUCZOWY na Windowsie
    train_model()